In [3]:
import os
import glob
import pandas as pd 
dataset_map = {'daphnet':1, 'to-vr':2, 'imu' :3, 'multimodal' : 4, 'oday' : 5, 'rempark' : 6, 'tdcs' : 7, 'weargait': 8}
dataset_units = {'daphnet':('mg', None), 'to-vr':('g','dps'), 'imu' :('g', 'dps'), 'multimodal' : ('g', 'dps'), 'oday' : ('m/s2','rad/s'), 'rempark' : ('g',None), 'tdcs' : ('m/s2', None)}

In [4]:
def convert_units(df: pd.DataFrame, acc: str = None, vang: str = None):
    df = df.copy()

    # --- Acceleration conversion ---
    # Convert acceleration to mg
    if acc in ['g', 'm/s2']:
        for col in df.columns:
            if 'acc' in col.lower():
                if acc == 'g':
                    # 1 g = 1000 mg
                    df[col] = df[col] * 1000  

                elif acc == 'm/s2':
                    # 1 m/s² = 101.971621 mg
                    df[col] = df[col] * 101.971621  

    # --- Angular velocity conversion ---
    # Convert rad/s to deg/s (dps)
    if vang == 'rad/s':
        for col in df.columns:
            if 'gyr' in col.lower():
                df[col] = df[col] * (180 / 3.141592653589793)

    return df


Daphnet

In [5]:
files = glob.glob('Daphnet/dataset_fog_release/dataset/*.txt')
n_dataset = dataset_map['daphnet']
colnames = ['timestamp',
            'shank_acc_x', 'shank_acc_y','shank_acc_z',
            'thigh_acc_x','thigh_acc_y','thigh_acc_z',
            'lowerback_acc_x','lowerback_acc_y', 'lowerback_acc_z',
            'FOG']
daphnet_l = [] 
for f in files: 
    p = f.split("S")[1][:2]
    t = f.split('R')[1][:2]
    df = pd.read_csv(f, sep=" ", header=None, names=colnames) 
    df['Task'] = t
    df['Patient'] = p
    df['Session'] = 1
    df['Dataset'] = n_dataset
    daphnet_l.append(df) 
    del df    

daphnet = pd.concat(daphnet_l)

daphnet = daphnet[daphnet['FOG'] != 0 ] 
daphnet['FOG'] = [0 if i == 1 else 1 for i in daphnet['FOG']]
del daphnet_l, p, t
daphnet

,timestamp,shank_acc_x,shank_acc_y,shank_acc_z,thigh_acc_x,thigh_acc_y,thigh_acc_z,lowerback_acc_x,lowerback_acc_y,lowerback_acc_z,FOG,Task,Patient,Session,Dataset
17919,280000,101,1000,297,-9,953,303,330,942,-145,0,02,06,1,1
17920,280015,101,1000,287,-27,944,292,310,933,-155,0,02,06,1,1
17921,280031,121,980,287,-27,935,292,300,952,-135,0,02,06,1,1
17922,280046,111,1000,277,-27,962,282,320,961,-165,0,02,06,1,1
17923,280062,101,980,306,-45,953,262,310,933,-126,0,02,06,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44795,699937,10,1029,108,-854,481,141,-106,495,-864,0,02,07,1,1
44796,699953,30,1049,108,-845,481,121,-97,495,-844,0,02,07,1,1
44797,699968,30,1019,108,-854,472,161,-106,495,-864,0,02,07,1,1
44798,699984,20,1039,118,-854,472,141,-106,495,-854,0,02,07,1,1


In [6]:
daphnet.to_csv('Structured_datasets/Daphnet.csv', mode= 'w')

IMU

In [7]:
files = glob.glob('IMU/IMU/*.txt')

In [8]:
IMU_l = [] 
n_dataset = dataset_map['imu']
acc = dataset_units['imu'][0] 
vang = dataset_units['imu'][1] 

for f in files: 
    p = f.split("SUB")[1][:2]
    t = f.split('_')[1][:1]
    df = pd.read_csv(f, sep = "\t") 
    df['Task'] = t
    df['Patient'] = p
    df['Session'] = 1
    df['Dataset'] = n_dataset
    IMU_l.append(df) 
    del df    

IMU = pd.concat(IMU_l)
del IMU_l, p, t

IMU.columns = ['Frame', 'Timestamps',	'ankle_acc_ml','ankle_acc_ap', 'ankle_acc_si', 'ankle_gyro_ml','ankle_gyro_ap', 'ankle_gyro_si', 'Task', 'Patient', 'Session', 'Dataset', 'FOG']
IMU

,Frame,Timestamps,ankle_acc_ml,ankle_acc_ap,ankle_acc_si,ankle_gyro_ml,ankle_gyro_ap,ankle_gyro_si,Task,Patient,Session,Dataset,FOG
0,1,0.007812,0.092264,-0.068927,0.985778,-2.034481,-3.055053,-1.909350,s,30,1,3,NaN
1,2,0.015625,0.116925,-0.044919,0.986332,-1.417150,-4.155275,-2.133970,s,30,1,3,NaN
2,3,0.023438,0.101301,-0.056452,0.984494,-1.232224,-5.330360,-0.982387,s,30,1,3,NaN
3,4,0.031250,0.087421,-0.058157,0.984171,-1.077129,-5.524511,-0.485618,s,30,1,3,NaN
4,5,0.039062,0.125733,-0.047518,0.982563,-0.457505,-5.748717,-1.221334,s,30,1,3,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15355,15356,119.968750,-0.933503,-0.398003,1.391941,64.648661,86.779523,-5.417468,1,05,1,3,0.0
15356,15357,119.976562,-0.444777,-0.490984,1.353721,76.473228,91.726520,-33.033496,1,05,1,3,0.0
15357,15358,119.984375,0.109763,-0.264055,1.441690,93.845202,93.828952,-7.382132,1,05,1,3,0.0
15358,15359,119.992188,0.000245,0.179466,1.131439,77.599476,89.149056,49.036361,1,05,1,3,0.0


In [9]:
IMU = IMU[IMU['Task']!= 's']
IMU= convert_units(IMU, acc, vang)
IMU.drop(columns = 'Frame', inplace = True)
IMU.to_csv('Structured_datasets/IMU.csv', mode= 'w')

del IMU 


MULTIMODAL

In [10]:
os.chdir('Multimodal/Filtered Data')
patients = glob.glob('*')

In [11]:
Multimodal_l = []
column_names = [
    # 1st column: Timestamps
    "Timestamps",
    
    # Columns 2-26: EEG signals
    "FP1", "FP2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", 
    "F7", "F8", "P7", "P8", "Fz", "Cz", "Pz", "FC1", "FC2", 
    "CP1", "CP2", "FC5", "FC6", "CP5", "CP6",
    
    # Columns 27-31: EMG, ECG, and Electrooculogram signals
    "EMG", "ECG", "EOG1", "EOG2", "EOG3",
    
    # Columns 32-59: Acceleration data (7 columns for each sensor)
    # Left shank
    "shank_l_acc_x", "shank_l_acc_y", "shank_l_acc_z",
    "shank_l_gyro_x", "shank_l_gyro_y", "shank_l_gyro_z", "shank_l_NC",
    # Right shank
    "shank_r_acc_x", "shank_r_acc_y", "shank_r_acc_z",
    "shank_r_gyro_x", "shank_r_gyro_y", "shank_r_gyro_z", "shank_r_NC",
    # Waist
    "waist_acc_x", "waist_acc_y", "waist_acc_z",
    "waist_gyro_x", "waist_gyro_y", "waist_gyro_z", "waist_NC",
    # Arm
    "arm_acc_x", "arm_acc_y", "arm_acc_z",
    "arm_gyro_x", "arm_gyro_y", "arm_gyro_z", "arm_SC",
    
    # Column 60: Label
    "FOG"
]
n_dataset=dataset_map['multimodal'] 
acc = dataset_units['multimodal'][0] 
vang = dataset_units['multimodal'][1] 

for p in patients: 
    os.chdir(p)
    files = glob.glob('*.txt')
    for f in files:
        t = f.split('_')[1][:1]
        df = pd.read_csv(f, sep = ",", header=None, index_col=0, names=column_names)
        df['Task'] = t
        df['Patient'] = p
        df['Session'] = 1
        df['Dataset'] = n_dataset 
        Multimodal_l.append(df) 
    os.chdir("..")


Multimodal = pd.concat(Multimodal_l)
del Multimodal_l, t, column_names, p

columns_to_filter = Multimodal.columns[1:-5]
columns_to_filter = [c for c in columns_to_filter if "gyro" in c or "acc" in c]
col_to_keep = ['Timestamps'] + columns_to_filter + list(Multimodal.columns[-5:])
Multimodal = Multimodal[col_to_keep]
del col_to_keep, columns_to_filter

Multimodal

,Timestamps,shank_l_acc_x,shank_l_acc_y,shank_l_acc_z,shank_l_gyro_x,shank_l_gyro_y,shank_l_gyro_z,shank_r_acc_x,shank_r_acc_y,shank_r_acc_z,...,arm_acc_y,arm_acc_z,arm_gyro_x,arm_gyro_y,arm_gyro_z,FOG,Task,Patient,Session,Dataset
0,09:44:36,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8208.000000,70.000000,-2281.000000,...,5751.000000,2996.000000,-12.000000,25.000000,18.000000,0,4,012,1,4
1,09:44:36.002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8214.745286,60.519403,-2280.305722,...,5750.033699,2997.235840,-12.566566,25.444136,17.489706,0,4,012,1,4
2,09:44:36.004,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8223.624791,49.194742,-2283.118555,...,5749.988254,2995.683047,-13.133272,26.127217,16.819744,0,4,012,1,4
3,09:44:36.006,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8232.931653,38.110380,-2287.478528,...,5750.125960,2991.312334,-13.616695,26.888230,16.104930,0,4,012,1,4
4,09:44:36.008,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8240.959010,29.350679,-2291.425667,...,5749.709110,2984.094414,-13.933412,27.566162,15.460078,0,4,012,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180496,12:10:38.992,8100.139833,-1689.587782,-1911.703935,-109.426247,-7.821348,-8.044901,0.000000,0.000000,0.000000,...,7361.136505,3317.507941,98.715799,76.215742,-7.082927,0,1,001,1,4
180497,12:10:38.994,8094.448841,-1690.430406,-1910.674232,-108.721276,-6.849944,-7.856461,0.000000,0.000000,0.000000,...,7358.299737,3316.900268,99.606686,77.931527,-7.179571,0,1,001,1,4
180498,12:10:38.996,8089.087934,-1691.279140,-1908.992562,-107.903181,-6.167867,-7.545571,0.000000,0.000000,0.000000,...,7351.694716,3316.438623,100.539672,79.239440,-7.234752,0,1,001,1,4
180499,12:10:38.998,8085.218018,-1691.885249,-1906.740595,-106.990057,-5.857193,-7.223121,0.000000,0.000000,0.000000,...,7343.526463,3316.384653,101.381772,80.231569,-7.193288,0,1,001,1,4


In [12]:
os.chdir('../..')
Multimodal= convert_units(Multimodal, acc, vang)
Multimodal.to_csv('Structured_datasets/Multimodal.csv', mode = 'w')
del Multimodal

TDCS

In [13]:
os.chdir('TDCS')

In [14]:
df = pd.read_parquet("tdcsfog_series_data.parquet", engine="pyarrow")
df.columns = ['Timestamps', 'lowerback_acc_v', 'lowerback_acc_ml', 'lowerback_acc_ap','St','Tu','Wa','Session','Patient','Series']
df['FOG'] = df[['St', 'Tu', 'Wa']].any(axis=1).astype(int)
df['Dataset'] = dataset_map['tdcs']
df['Task'] = 1
df = df[['Timestamps', 'lowerback_acc_v', 'lowerback_acc_ml', 'lowerback_acc_ap','FOG','Task','Session','Patient','Dataset']]
df

,Timestamps,lowerback_acc_v,lowerback_acc_ml,lowerback_acc_ap,FOG,Task,Session,Patient,Dataset
0,0,-9.533939,0.566322,-1.413525,0,1,003f117e14,13abfd,7
1,1,-9.536140,0.564137,-1.440621,0,1,003f117e14,13abfd,7
2,2,-9.529346,0.561765,-1.429332,0,1,003f117e14,13abfd,7
3,3,-9.531239,0.564227,-1.415490,0,1,003f117e14,13abfd,7
4,4,-9.540825,0.561854,-1.429471,0,1,003f117e14,13abfd,7
...,...,...,...,...,...,...,...,...,...
4220,4220,-9.403467,0.089003,-3.220304,0,1,ffda8fadfd,bae0ce,7
4221,4221,-9.404245,0.090531,-3.216584,0,1,ffda8fadfd,bae0ce,7
4222,4222,-9.405770,0.084380,-3.224039,0,1,ffda8fadfd,bae0ce,7
4223,4223,-9.403579,0.084236,-3.236686,0,1,ffda8fadfd,bae0ce,7


In [15]:
os.chdir('..')
acc = dataset_units['tdcs'][0] 
vang = dataset_units['tdcs'][1] 
df = convert_units(df, acc, vang)
#df.to_csv('Structured_datasets/tdcs.csv', mode = 'w')

del df

oday

In [16]:
import os
import glob
import pandas as pd 
os.chdir('imu-fog-detection/data/raw/imus6_subjects7/')
files= glob.glob('*')

In [17]:
oday_l = [] 
n_dataset = dataset_map['oday']
for f in files: 
    p = f.split("_")[0][2]
    t = f.split('_')[5]
    s = f.split('_')[2]
    df = pd.read_excel(f) 
    df = df.drop('subject_ID', axis=1)
    df.columns = ['Timestamps', 
                  'chest_acc_x', 'chest_acc_y', 'chest_acc_z', 
                  'chest_gyro_x', 'chest_gyro_y', 'chest_gyro_z', 
                  
                  'lowerback_acc_x', 'lowerback_acc_y', 'lowerback_acc_z', 
                  'lowerback_gyro_x', 'lowerback_gyro_y', 'lowerback_gyro_z', 
                  
                  'ankle_l_acc_x', 'ankle_l_acc_y', 'ankle_l_acc_z', 
                  'ankle_l_gyro_x', 'ankle_l_gyro_y', 'ankle_l_gyro_z', 
                  
                  'ankle_r_acc_x', 'ankle_r_acc_y', 'ankle_r_acc_z', 
                  'ankle_r_gyro_x', 'ankle_r_gyro_y', 'ankle_r_gyro_z',
                  
                  'foot_l_acc_x', 'foot_l_acc_y', 'foot_l_acc_z', 
                  'foot_l_gyro_x', 'foot_l_gyro_y', 'foot_l_gyro_z',
                  
                  'foot_r_acc_x', 'foot_r_acc_y', 'foot_r_acc_z', 
                  'foot_r_gyro_x', 'foot_r_gyro_y', 'foot_r_gyro_z', 
                  
                  'head_acc_x', 'head_acc_y', 'head_acc_z', 
                  'head_gyro_x', 'head_gyro_y', 'head_gyro_z',
                  
                  'thigh_l_acc_x', 'thigh_l_acc_y', 'thigh_l_acc_z', 
                  'thigh_l_gyro_x', 'thigh_l_gyro_y', 'thigh_l_gyro_z', 
                  
                  'thigh_r_acc_x', 'thigh_r_acc_y', 'thigh_r_acc_z',
                  'thigh_r_gyro_x', 'thigh_r_gyro_y', 'thigh_r_gyro_z',
                  
                  'wrist_l_acc_x', 'wrist_l_acc_y', 'wrist_l_acc_z', 
                  'wrist_l_gyro_x', 'wrist_l_gyro_y', 'wrist_l_gyro_z', 
                  
                  'wrist_r_acc_x', 'wrist_r_acc_y', 'wrist_r_acc_z', 
                  'wrist_r_gyro_x', 'wrist_r_gyro_y', 'wrist_r_gyro_z',
                  
                  'FOG' ]
    df['Task'] = t
    df['Patient'] = p
    df['Session'] = s
    df['Dataset'] = n_dataset
    oday_l.append(df) 
    del df  

oday = pd.concat(oday_l) 
del oday_l, p, t, s 
acc = dataset_units['oday'][0] 
vang = dataset_units['oday'][1] 
oday = convert_units(oday, acc, vang)

In [18]:
os.chdir('../../../..')
oday.to_csv('Structured_datasets/Oday.csv', mode = 'w')


to-vr

In [39]:
os.chdir('..')
os.chdir('IEEE 2024')

In [40]:
df = pd.read_csv('final_dataset.csv')

df.columns = df.columns.str.replace('left', 'l', regex=True)
df.columns = df.columns.str.replace('right', 'r', regex=True)
df.columns = df.columns.str.replace('Unnamed: 0', 'Timestamps', regex=True)
df['Dataset'] = dataset_map['to-vr']
acc = dataset_units['to-vr'][0] 
vang = dataset_units['to-vr'][1] 

In [41]:
os.chdir('..')

In [42]:
os.chdir('Datasets_public_FOG/Structured_datasets')
df = convert_units(df, acc, vang)
df

,Timestamps,ankle_l_acc_x,ankle_l_acc_y,ankle_l_acc_z,ankle_l_gyro_x,ankle_l_gyro_y,ankle_l_gyro_z,ankle_r_acc_x,ankle_r_acc_y,ankle_r_acc_z,...,wrist_gyro_z,activity-Torino,activity-Verona,fog-Agree,fog-Torino,fog-Verona,patient,session,task,Dataset
0,2022-05-25 13:51:01.028146,-1014.984493,133.385822,24.818359,0.212810,1.589006,-0.167686,-1005.513712,-97.270321,13.287343,...,0.327340,2,2,0,0,0,107,1,11,2
1,2022-05-25 13:51:01.044816,-1000.777415,136.812256,34.778213,-0.074904,1.756708,-0.114430,-1007.110082,-95.888491,6.186690,...,0.717459,2,2,0,0,0,107,1,11,2
2,2022-05-25 13:51:01.061486,-1005.732003,130.559716,32.259090,0.183994,1.572684,-0.083382,-1004.861259,-99.394543,7.403197,...,0.300301,2,2,0,0,0,107,1,11,2
3,2022-05-25 13:51:01.078156,-1003.625605,129.368818,35.319069,-0.066717,1.803254,-0.083298,-1003.089357,-98.309608,10.601766,...,0.525266,2,2,0,0,0,107,1,11,2
4,2022-05-25 13:51:01.094826,-1004.711778,140.318037,29.175567,0.020656,1.400361,-0.203291,-1001.984964,-98.685610,12.207213,...,0.466261,2,2,0,0,0,107,1,11,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329022,2021-07-28 12:47:18.685109,NaN,NaN,NaN,NaN,NaN,NaN,-981.035903,77.964258,-114.742657,...,-2.041297,0,3,0,0,0,900816405,1,7,2
329023,2021-07-28 12:47:18.701779,NaN,NaN,NaN,NaN,NaN,NaN,-993.543182,73.625378,-99.369733,...,-2.316178,0,3,0,0,0,900816405,1,7,2
329024,2021-07-28 12:47:18.718449,NaN,NaN,NaN,NaN,NaN,NaN,-998.043626,66.761259,-106.356999,...,-2.196800,0,3,0,0,0,900816405,1,7,2
329025,2021-07-28 12:47:18.735119,NaN,NaN,NaN,NaN,NaN,NaN,-996.486452,66.008835,-102.997380,...,-2.315260,0,3,0,0,0,900816405,1,7,2


In [43]:
df['FOG'] = [0 if i == 0 else 1 for i in df['fog-Agree']]
df = df.drop(['fog-Agree', 'fog-Torino', 'fog-Verona', 'activity-Verona','activity-Torino'], axis = 1) 
df

,Timestamps,ankle_l_acc_x,ankle_l_acc_y,ankle_l_acc_z,ankle_l_gyro_x,ankle_l_gyro_y,ankle_l_gyro_z,ankle_r_acc_x,ankle_r_acc_y,ankle_r_acc_z,...,wrist_acc_y,wrist_acc_z,wrist_gyro_x,wrist_gyro_y,wrist_gyro_z,patient,session,task,Dataset,FOG
0,2022-05-25 13:51:01.028146,-1014.984493,133.385822,24.818359,0.212810,1.589006,-0.167686,-1005.513712,-97.270321,13.287343,...,-318.554297,428.788538,0.592055,-0.191380,0.327340,107,1,11,2,0
1,2022-05-25 13:51:01.044816,-1000.777415,136.812256,34.778213,-0.074904,1.756708,-0.114430,-1007.110082,-95.888491,6.186690,...,-306.927971,428.818868,0.734625,-0.414149,0.717459,107,1,11,2,0
2,2022-05-25 13:51:01.061486,-1005.732003,130.559716,32.259090,0.183994,1.572684,-0.083382,-1004.861259,-99.394543,7.403197,...,-309.479757,413.886357,0.426720,-0.243212,0.300301,107,1,11,2,0
3,2022-05-25 13:51:01.078156,-1003.625605,129.368818,35.319069,-0.066717,1.803254,-0.083298,-1003.089357,-98.309608,10.601766,...,-308.846981,418.330086,0.759547,-0.233320,0.525266,107,1,11,2,0
4,2022-05-25 13:51:01.094826,-1004.711778,140.318037,29.175567,0.020656,1.400361,-0.203291,-1001.984964,-98.685610,12.207213,...,-307.129322,431.260938,0.676813,-0.193347,0.466261,107,1,11,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329022,2021-07-28 12:47:18.685109,NaN,NaN,NaN,NaN,NaN,NaN,-981.035903,77.964258,-114.742657,...,-382.555492,495.521045,2.710197,1.893279,-2.041297,900816405,1,7,2,0
329023,2021-07-28 12:47:18.701779,NaN,NaN,NaN,NaN,NaN,NaN,-993.543182,73.625378,-99.369733,...,-370.881378,483.679458,0.777528,1.536229,-2.316178,900816405,1,7,2,0
329024,2021-07-28 12:47:18.718449,NaN,NaN,NaN,NaN,NaN,NaN,-998.043626,66.761259,-106.356999,...,-389.143985,487.086735,-0.649782,1.492471,-2.196800,900816405,1,7,2,0
329025,2021-07-28 12:47:18.735119,NaN,NaN,NaN,NaN,NaN,NaN,-996.486452,66.008835,-102.997380,...,-402.613005,475.472304,-0.232398,1.805738,-2.315260,900816405,1,7,2,0


In [44]:
df = df.rename(columns={
    'task': 'Task',
    'patient': 'Patient',
    'session': 'Session'
})

df.to_csv('FOGSTAR.csv', mode = 'w')

In [28]:
os.chdir('..')
rem = pd.read_csv('rempark_raw.csv')

In [29]:
rem.columns = ['waist_acc_x', 'waist_acc_y', 'waist_acc_z', 'FOG', 'Patient', 'Session'] 
rem['Task'] = 1
rem['Dataset'] = dataset_map['rempark'] 
rem

,waist_acc_x,waist_acc_y,waist_acc_z,FOG,Patient,Session,Task,Dataset
0,-0.082092,-0.229857,5.385219,0,1,1,1,6
1,-0.168878,-0.456441,11.045588,0,1,1,1,6
2,-0.141783,-0.363217,9.282669,0,1,1,1,6
3,-0.121982,-0.375582,9.090811,0,1,1,1,6
4,-0.133264,-0.411417,9.875754,0,1,1,1,6
...,...,...,...,...,...,...,...,...
2684355,2.967190,-4.634104,9.207251,0,15,3,1,6
2684356,3.031686,-4.304834,9.269807,0,15,3,1,6
2684357,2.769231,-4.280510,9.026146,0,15,3,1,6
2684358,2.748902,-4.528128,9.087254,0,15,3,1,6


In [31]:
os.chdir('Structured_datasets')
acc = dataset_units['rempark'][0]
vang = dataset_units['rempark'][1]
rem = convert_units(rem, acc, vang) 
rem.to_csv('rempark.csv', mode='w')